In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

**Reload Embedding**

In [1]:
import numpy as np

X_train = np.load("/kaggle/input/newsclassifier-model/X_train.npy")
X_test = np.load("/kaggle/input/newsclassifier-model/X_test.npy")
y_train = np.load("/kaggle/input/newsclassifier-model/y_train.npy")
y_test = np.load("/kaggle/input/newsclassifier-model/y_test.npy")

print("Embeddings Reloaded")

Embeddings Reloaded


In [2]:
from sentence_transformers import SentenceTransformer

# Load embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

2025-05-14 05:44:01.647193: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747201441.836211      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747201441.889159      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
X_train.shape

(120000, 384)

**LSTM**

In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

# Reshape input to (batch_size, time_steps, input_dim) → (24, 16)
X_train_reshaped = X_train.reshape(-1, 24, 16).astype(np.float32)
X_test_reshaped = X_test.reshape(-1, 24, 16).astype(np.float32)

# Ensure labels are int32
y_train = y_train.astype(np.int32)
y_test = y_test.astype(np.int32)

# Define the LSTM model
model = models.Sequential([
    layers.LSTM(128, input_shape=(24, 16), return_sequences=False),
    layers.Dense(4, activation='softmax')  # 4 classes for AG News
])

# Compile the model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Train the model
model.fit(X_train_reshaped, y_train, epochs=5, batch_size=64, validation_data=(X_test_reshaped, y_test))

# Evaluate the model
loss, accuracy = model.evaluate(X_test_reshaped, y_test, batch_size=64)
print(f"LSTM Accuracy: {accuracy * 100:.2f}%")

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.6374 - loss: 0.8826 - val_accuracy: 0.8075 - val_loss: 0.5235
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.8235 - loss: 0.4860 - val_accuracy: 0.8366 - val_loss: 0.4638
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.8525 - loss: 0.4155 - val_accuracy: 0.8530 - val_loss: 0.4094
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.8640 - loss: 0.3848 - val_accuracy: 0.8659 - val_loss: 0.3763
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.8701 - loss: 0.3664 - val_accuracy: 0.8734 - val_loss: 0.3501
119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8676 - loss: 0.3628
LSTM Accuracy: 87.34%


**Test Case**

In [8]:
import numpy as np

def predict_news_category_lstm(news_text):
    labels = ["World", "Sports", "Business", "Sci/Tech"]
    
    # Encode the input text
    embedded = embedder.encode([news_text])  # shape: (1, 384)

    # Reshape for LSTM input: (batch_size, time_steps=24, input_dim=16)
    lstm_input = np.array(embedded).reshape(1, 24, 16).astype(np.float32)

    # Predict
    predictions = model.predict(lstm_input)
    predicted_class = np.argmax(predictions, axis=1)[0]
    
    return labels[predicted_class]

# Example usage
custom_news = input("Enter a news: ")
print("Predicted Category (LSTM):", predict_news_category_lstm(custom_news))

Enter a news:  Today i have a deep learning class


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
Predicted Category (LSTM): Sci/Tech
